# BIPIA EmailQA Indirect Prompt Injection Detection

This notebook is designed to run from the root directory of the GitHub repository. It does not depend on a personal Google Drive path.

In [ ]:
import sys

print("Python version:", sys.version)
print("Python is working correctly.")

In [ ]:
from pathlib import Path

# Run this notebook from the repository root
PROJECT_DIR = Path.cwd().resolve()
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
EMAIL_DIR = RAW_DIR / 'email'
ATTACKS_DIR = RAW_DIR / 'attacks'
OUTPUT_DIR = PROJECT_DIR / 'outputs'

required_files = [
    EMAIL_DIR / 'train.jsonl',
    EMAIL_DIR / 'test.jsonl',
    ATTACKS_DIR / 'text_attack_train.json',
    ATTACKS_DIR / 'text_attack_test.json',
]
missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    missing_list = ', '.join(str(path) for path in missing_files)
    raise FileNotFoundError(
        'Required BIPIA source files are missing. Run the notebook from the repository root and ensure data/raw contains the required files: '
        + missing_list
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project directory:', PROJECT_DIR)
print('Output directory:', OUTPUT_DIR)

In [ ]:
files_to_check = {
    "Email train": EMAIL_DIR / "train.jsonl",
    "Email test": EMAIL_DIR / "test.jsonl",
    "Attack train": ATTACKS_DIR / "text_attack_train.json",
    "Attack test": ATTACKS_DIR / "text_attack_test.json",
}

for name, path in files_to_check.items():
    print(f"{name}: {path.exists()} — {path.name}")

In [ ]:
import json

# قراءة بيانات البريد بصيغة JSONL
def load_jsonl(file_path):
    records = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as error:
                    print(f"Error in line {line_number}: {error}")
                    raise

    return records


email_train = load_jsonl(EMAIL_DIR / "train.jsonl")
email_test = load_jsonl(EMAIL_DIR / "test.jsonl")

# قراءة ملفات الهجمات بصيغة JSON
with open(
    ATTACKS_DIR / "text_attack_train.json",
    "r",
    encoding="utf-8"
) as file:
    attacks_train = json.load(file)

with open(
    ATTACKS_DIR / "text_attack_test.json",
    "r",
    encoding="utf-8"
) as file:
    attacks_test = json.load(file)

print("Files loaded successfully.")

In [ ]:
print("Email train records:", len(email_train))
print("Email test records:", len(email_test))

print("Attack train type:", type(attacks_train).__name__)
print("Attack test type:", type(attacks_test).__name__)

print("Attack train size:", len(attacks_train))
print("Attack test size:", len(attacks_test))

In [ ]:
print("Email record type:", type(email_train[0]).__name__)
print("Email fields:", list(email_train[0].keys()))

print("\nAttack train top-level keys:")
print(list(attacks_train.keys()))

print("\nAttack test top-level keys:")
print(list(attacks_test.keys()))

In [ ]:
print("Attack train value types:")

for key, value in attacks_train.items():
    length = len(value) if hasattr(value, "__len__") else "N/A"

    print(
        f"{key}: "
        f"type={type(value).__name__}, "
        f"length={length}"
    )

In [ ]:
first_key = next(iter(attacks_train))
first_value = attacks_train[first_key]

print("First category:", first_key)
print("Category value type:", type(first_value).__name__)
print("Number of items:", len(first_value))
print("First item type:", type(first_value[0]).__name__)

if isinstance(first_value[0], dict):
    print("First item fields:", list(first_value[0].keys()))

In [ ]:
import pandas as pd

def inspect_email_records(records, split_name):
    df = pd.DataFrame(records)

    print(f"\n--- {split_name} ---")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    print("Missing values:")
    print(df.isna().sum())

    for column in ["context", "question", "ideal"]:
        print(
            f"{column} types:",
            df[column].map(type).value_counts().to_dict()
        )

    context_lengths = df["context"].str.len()

    print("Context minimum length:", context_lengths.min())
    print("Context median length:", int(context_lengths.median()))
    print("Context maximum length:", context_lengths.max())


inspect_email_records(email_train, "Email train")
inspect_email_records(email_test, "Email test")

In [ ]:
def flatten_attacks(attack_dictionary):
    rows = []

    for category, attack_list in attack_dictionary.items():
        for attack_number, attack_text in enumerate(
            attack_list,
            start=1
        ):
            rows.append({
                "category": category,
                "attack_number": attack_number,
                "attack_text": attack_text
            })

    return pd.DataFrame(rows)


attack_train_df = flatten_attacks(attacks_train)
attack_test_df = flatten_attacks(attacks_test)

print("Attack train shape:", attack_train_df.shape)
print("Attack test shape:", attack_test_df.shape)

print(
    "Train attack text types:",
    attack_train_df["attack_text"].map(type).value_counts().to_dict()
)

print(
    "Test attack text types:",
    attack_test_df["attack_text"].map(type).value_counts().to_dict()
)

print(
    "Train attack missing values:",
    attack_train_df["attack_text"].isna().sum()
)

print(
    "Test attack missing values:",
    attack_test_df["attack_text"].isna().sum()
)

print(
    "Train attack duplicates:",
    attack_train_df["attack_text"].duplicated().sum()
)

print(
    "Test attack duplicates:",
    attack_test_df["attack_text"].duplicated().sum()
)

In [ ]:
def build_binary_dataset(
    email_records,
    attack_dictionary,
    split_name
):
    attack_categories = list(attack_dictionary.keys())
    rows = []

    for context_index, email_record in enumerate(email_records):
        context_text = email_record["context"].strip()

        # توزيع منتظم وحتمي على فئات الهجمات
        category_index = context_index % len(attack_categories)
        attack_category = attack_categories[category_index]

        category_attacks = attack_dictionary[attack_category]

        attack_index = (
            context_index // len(attack_categories)
        ) % len(category_attacks)

        attack_text = category_attacks[attack_index].strip()

        context_id = f"{split_name}_{context_index:03d}"

        # النسخة السليمة
        rows.append({
            "sample_id": f"{context_id}_benign",
            "context_id": context_id,
            "split": split_name,
            "text": context_text,
            "label": 0,
            "label_name": "benign",
            "attack_category": None,
            "attack_number": None
        })

        # النسخة المصابة
        attacked_text = context_text + "\n\n" + attack_text

        rows.append({
            "sample_id": f"{context_id}_attack",
            "context_id": context_id,
            "split": split_name,
            "text": attacked_text,
            "label": 1,
            "label_name": "attack",
            "attack_category": attack_category,
            "attack_number": attack_index + 1
        })

    return pd.DataFrame(rows)


train_df = build_binary_dataset(
    email_records=email_train,
    attack_dictionary=attacks_train,
    split_name="train"
)

test_df = build_binary_dataset(
    email_records=email_test,
    attack_dictionary=attacks_test,
    split_name="test"
)

print("Binary datasets created successfully.")

In [ ]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain label distribution:")
print(train_df["label_name"].value_counts())

print("\nTest label distribution:")
print(test_df["label_name"].value_counts())

print("\nTrain unique contexts:")
print(train_df["context_id"].nunique())

print("\nTest unique contexts:")
print(test_df["context_id"].nunique())

print("\nMissing text values:")
print("Train:", train_df["text"].isna().sum())
print("Test:", test_df["text"].isna().sum())

print("\nDuplicate sample IDs:")
print("Train:", train_df["sample_id"].duplicated().sum())
print("Test:", test_df["sample_id"].duplicated().sum())

In [ ]:
# Check for exact full-text overlap between training and test sets
train_texts = set(train_df["text"].astype(str))
test_texts = set(test_df["text"].astype(str))

full_text_overlap = train_texts.intersection(test_texts)

print("Number of overlapping full texts:", len(full_text_overlap))

if len(full_text_overlap) == 0:
    print("PASS: No full-text overlap between train and test.")
else:
    print("WARNING: Overlapping full texts found:")
    for text in list(full_text_overlap)[:3]:
        print(text[:300])
        print("-" * 60)

# Check overlap of original clean email contexts only
train_benign_texts = set(
    train_df.loc[train_df["label_name"] == "benign", "text"].astype(str)
)

test_benign_texts = set(
    test_df.loc[test_df["label_name"] == "benign", "text"].astype(str)
)

benign_context_overlap = train_benign_texts.intersection(test_benign_texts)

print("\nNumber of overlapping original email contexts:", len(benign_context_overlap))

if len(benign_context_overlap) == 0:
    print("PASS: No original email-context overlap between train and test.")
else:
    print("WARNING: Overlapping original contexts found:")
    for text in list(benign_context_overlap)[:3]:
        print(text[:300])
        print("-" * 60)

In [ ]:
# Identify original clean email contexts shared with the training set
train_benign_texts = set(
    train_df.loc[train_df["label_name"] == "benign", "text"].astype(str)
)

# Find test context IDs whose clean version overlaps with training
overlapping_test_context_ids = test_df.loc[
    (test_df["label_name"] == "benign") &
    (test_df["text"].astype(str).isin(train_benign_texts)),
    "context_id"
].unique()

print("Overlapping test contexts to remove:", len(overlapping_test_context_ids))

# Remove both benign and attack samples belonging to overlapping contexts
test_df_clean = test_df.loc[
    ~test_df["context_id"].isin(overlapping_test_context_ids)
].copy()

print("\nCleaned test shape:", test_df_clean.shape)
print("\nCleaned test label distribution:")
print(test_df_clean["label_name"].value_counts())

In [ ]:
# Verify that the cleaned test set has no overlap with training
train_texts = set(train_df["text"].astype(str))
clean_test_texts = set(test_df_clean["text"].astype(str))

full_text_overlap = train_texts.intersection(clean_test_texts)

train_benign_texts = set(
    train_df.loc[train_df["label_name"] == "benign", "text"].astype(str)
)
clean_test_benign_texts = set(
    test_df_clean.loc[
        test_df_clean["label_name"] == "benign", "text"
    ].astype(str)
)

benign_context_overlap = train_benign_texts.intersection(
    clean_test_benign_texts
)

print("Number of overlapping full texts:", len(full_text_overlap))
print("Number of overlapping original email contexts:", len(benign_context_overlap))

assert len(full_text_overlap) == 0
assert len(benign_context_overlap) == 0

print("\nPASS: The cleaned test set has no text or context overlap with training.")

In [ ]:
# Create a directory for the final processed datasets
PROCESSED_DIR = OUTPUT_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Define output paths
train_output_path = PROCESSED_DIR / "binary_train.csv"
test_output_path = PROCESSED_DIR / "binary_test_clean.csv"

# Save the leakage-free datasets
train_df.to_csv(train_output_path, index=False, encoding="utf-8")
test_df_clean.to_csv(test_output_path, index=False, encoding="utf-8")

print("Training dataset saved to:", train_output_path)
print("Cleaned test dataset saved to:", test_output_path)

# Reload files to verify successful saving
saved_train_df = pd.read_csv(train_output_path)
saved_test_df = pd.read_csv(test_output_path)

print("\nSaved training shape:", saved_train_df.shape)
print("Saved cleaned test shape:", saved_test_df.shape)

print("\nSaved training label distribution:")
print(saved_train_df["label_name"].value_counts())

print("\nSaved cleaned test label distribution:")
print(saved_test_df["label_name"].value_counts())

In [ ]:
import time
import pickle
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score
)

# Reload the final leakage-free datasets
PROCESSED_DIR = OUTPUT_DIR / "processed"

train_data = pd.read_csv(PROCESSED_DIR / "binary_train.csv")
test_data = pd.read_csv(PROCESSED_DIR / "binary_test_clean.csv")

# Use only the document text as input; exclude all metadata columns
TEXT_COLUMN = "text"
TARGET_COLUMN = "label"

X_train = train_data[TEXT_COLUMN].fillna("").astype(str)
y_train = train_data[TARGET_COLUMN]

X_test = test_data[TEXT_COLUMN].fillna("").astype(str)
y_test = test_data[TARGET_COLUMN]

# Create identical TF-IDF preprocessing for every model
def create_pipeline(classifier):
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            analyzer="word",
            ngram_range=(1, 2),
            norm="l2"
        )),
        ("classifier", classifier)
    ])

models = {
    "Dummy Baseline": create_pipeline(
        DummyClassifier(strategy="most_frequent")
    ),
    "Multinomial Naive Bayes": create_pipeline(
        MultinomialNB()
    ),
    "Logistic Regression": create_pipeline(
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    ),
    "Linear SVM": create_pipeline(
        LinearSVC(
            class_weight="balanced",
            random_state=42
        )
    )
}

results = []
trained_models = {}

for model_name, model in models.items():
    start_time = time.perf_counter()

    # Fit TF-IDF and classifier using training data only
    model.fit(X_train, y_train)

    training_time = time.perf_counter() - start_time
    y_pred = model.predict(X_test)

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    ).ravel()

    # Compute AUC from probability or decision scores
    if hasattr(model, "predict_proba"):
        attack_scores = model.predict_proba(X_test)[:, 1]
    else:
        attack_scores = model.decision_function(X_test)

    auc = roc_auc_score(y_test, attack_scores)
    model_size_kb = len(pickle.dumps(model)) / 1024

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Macro Precision": macro_precision,
        "Macro Recall": macro_recall,
        "Macro F1": macro_f1,
        "Attack Recall": tp / (tp + fn),
        "False Positive Rate": fp / (fp + tn),
        "AUC": auc,
        "Training Time (sec)": training_time,
        "Model Size (KB)": model_size_kb
    })

    trained_models[model_name] = model

results_df = pd.DataFrame(results).sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

display(results_df.round(3))

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix
)

# Select the best model based on Macro F1
best_model_name = "Linear SVM"
best_model = trained_models[best_model_name]

# Generate predictions on the leakage-free test set
best_predictions = best_model.predict(X_test)

# Print detailed classification metrics
print(f"Best model: {best_model_name}\n")

print(
    classification_report(
        y_test,
        best_predictions,
        target_names=["benign", "attack"],
        digits=3
    )
)

# Display the confusion matrix
confusion = confusion_matrix(y_test, best_predictions, labels=[0, 1])

display_plot = ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=["benign", "attack"]
)

display_plot.plot(cmap="Blues", values_format="d")
plt.title("Linear SVM Confusion Matrix on Unseen Attack Categories")
plt.show()

In [ ]:
# Extract the most influential TF-IDF features from the Linear SVM model
tfidf_vectorizer = best_model.named_steps["tfidf"]
svm_classifier = best_model.named_steps["classifier"]

feature_names = tfidf_vectorizer.get_feature_names_out()
feature_weights = svm_classifier.coef_[0]

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Weight": feature_weights
})

# Positive weights support the attack class
top_attack_features = feature_importance.nlargest(15, "Weight")

# Negative weights support the benign class
top_benign_features = feature_importance.nsmallest(15, "Weight")

print("Top features supporting the attack class:")
display(top_attack_features.reset_index(drop=True).round(3))

print("\nTop features supporting the benign class:")
display(top_benign_features.reset_index(drop=True).round(3))

In [ ]:
# Evaluate Linear SVM recall separately for each unseen attack category
test_predictions_df = test_data.copy()

test_predictions_df["predicted_label"] = best_predictions
test_predictions_df["is_correct"] = (
    test_predictions_df["label"] == test_predictions_df["predicted_label"]
)

# Analyze attack samples only; attack_category is used for reporting only
attack_category_results = (
    test_predictions_df.loc[
        test_predictions_df["label_name"] == "attack"
    ]
    .groupby("attack_category")
    .agg(
        Attack_Samples=("label", "size"),
        Detected_Attacks=("predicted_label", lambda x: (x == 1).sum()),
        Missed_Attacks=("predicted_label", lambda x: (x == 0).sum()),
        Attack_Recall=("predicted_label", lambda x: (x == 1).mean())
    )
    .sort_values("Attack_Recall", ascending=False)
)

display(attack_category_results.round(3))

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score

# Define the same four classifiers used in the main experiment
classifier_factories = {
    "Dummy Baseline": lambda: DummyClassifier(strategy="most_frequent"),
    "Multinomial Naive Bayes": lambda: MultinomialNB(),
    "Logistic Regression": lambda: LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),
    "Linear SVM": lambda: LinearSVC(
        class_weight="balanced",
        random_state=42
    )
}

# Compare unigram-only features against unigram-bigram features
feature_conditions = {
    "Unigrams only": (1, 1),
    "Unigrams + bigrams": (1, 2)
}

ablation_rows = []

for condition_name, ngram_range in feature_conditions.items():
    for model_name, classifier_factory in classifier_factories.items():

        pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(
                lowercase=True,
                analyzer="word",
                ngram_range=ngram_range,
                norm="l2"
            )),
            ("classifier", classifier_factory())
        ])

        # Fit preprocessing and classifier on training data only
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)

        ablation_rows.append({
            "Feature Condition": condition_name,
            "Model": model_name,
            "Accuracy": accuracy_score(y_test, predictions),
            "Macro F1": f1_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0
            ),
            "Attack Recall": recall_score(
                y_test,
                predictions,
                pos_label=1,
                zero_division=0
            )
        })

ablation_df = pd.DataFrame(ablation_rows)

ablation_pivot = ablation_df.pivot(
    index="Model",
    columns="Feature Condition",
    values="Macro F1"
)

display(ablation_df.round(3))
print("\nMacro-F1 comparison for the ablation:")
display(ablation_pivot.round(3))

In [ ]:
# Create a directory for all final experiment outputs
METRICS_DIR = OUTPUT_DIR / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Save the main model-comparison results
results_df.to_csv(
    METRICS_DIR / "main_model_comparison.csv",
    index=False,
    encoding="utf-8"
)

# Save robustness results for unseen attack categories
attack_category_results.reset_index().to_csv(
    METRICS_DIR / "unseen_attack_category_results.csv",
    index=False,
    encoding="utf-8"
)

# Save the unigram versus unigram-bigram ablation results
ablation_df.to_csv(
    METRICS_DIR / "ngram_ablation_results.csv",
    index=False,
    encoding="utf-8"
)

ablation_pivot.reset_index().to_csv(
    METRICS_DIR / "ngram_ablation_macro_f1.csv",
    index=False,
    encoding="utf-8"
)

# Save Linear SVM feature-importance results
top_attack_features.to_csv(
    METRICS_DIR / "linear_svm_top_attack_features.csv",
    index=False,
    encoding="utf-8"
)

top_benign_features.to_csv(
    METRICS_DIR / "linear_svm_top_benign_features.csv",
    index=False,
    encoding="utf-8"
)

print("Saved files:")
for file_path in sorted(METRICS_DIR.iterdir()):
    print("-", file_path.name)